# Barnes Analysis Pipeline
Interface for data visualization. Refer to `data_extractor.py` for more parameter plot adjustments.


In [ ]:
#provide instructions for running the script just like deeplabcut


import sys
sys.path.append('..') # Add parent directory to path to import modules

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from data_extractor import extract_data
from plotting import (
    all_csv, 
    all_csv_ab, 
    day_combined_scatter, 
    day_comp_scatter, 
    plot_daily_summary_bar, 
    plot_daily_traces_separated, 
    plot_mouse_daily_traces,
    plot_daily_tracesxs
)

## 1. Parameters & Data Extraction
Set the parameters for your analysis here. 

"ACTION" Parameter will be used to determine whether interaction selected is from when entering or exiting the hole

"ZSCORE" df/f relative to one another

"TRUNC_TIME" How long is the interaction occuring for

"LOOPTYPE" determines the comparison of phases in the day

DATA EXTRACTION just makes sure to prepare the data to be arranged for their own storage

In [ ]:
# Parameters
action = True  # True for Entrance, False for Exit
zscore = True
trunc_time = 20
Days = ["D1", "D2", "D3", "D4", "D5", "D6", "D7", "D8"]
Phases = ["PH1", "PH2", "PH3", "PH4", "TD"]
looptype = False # False means phases in day loop

# Run Extraction
print("Extracting data... Please wait.")
results = extract_data(action, zscore, trunc_time, looptype, Days, Phases)
print("Data extraction complete!")

# Unpack commonly used variables for the plotting cells
all_traces = results['all_traces']
all_amplitudes = results['all_amplitudes']
all_durations = results['all_durations']
all_widths = results['all_widths']
all_aucs = results['all_aucs']
all_aucs_rate = results['all_aucs_rate']
all_rates = results['all_rates']
mouse_description = results['mouse_description']
time_axis = results['time_axis']

## 2. Export CSVs
Run this cell to export CSV summaries of your data. The files will be saved in your configured output directories.

In [ ]:
print("Exporting CSVs...")
for day in Days:
    all_csv(all_aucs_rate, day, Phases, 1, unit="auc_rate", save_path=True, action=action, mouse_log=mouse_description)
    all_csv(all_rates, day, Phases, 1, unit="rate", save_path=False, action=action, mouse_log=mouse_description)
    
    all_csv_ab(all_aucs_rate, day, Phases, 1, unit="auc_rate", save_path=True, action=action, mouse_log=mouse_description)
    all_csv_ab(all_rates, day, Phases, 1, unit="rate", save_path=True, action=action, mouse_log=mouse_description)
print("Done!")

## 3. Scatter Plots
Generate combined and day-comparison scatter plots. You can tweak the `ax1lim` and `ax2lim` parameters here to adjust the axis limits of the generated plots.

In [ ]:

# visualization or saving the file 
save_plots = False

plt.close('all')

for day in Days:
    day_combined_scatter(all_aucs_rate, day, Phases, 1, unit="auc_rate", save_path=save_plots, action=action, mouse_log=mouse_description, ax1lim=(15,15), ax2lim=(20,20))
    day_combined_scatter(all_rates, day, Phases, 1, unit="rate", save_path=False, action=action)
    
    day_comp_scatter(all_aucs_rate, day, Phases, 1, unit="auc_rate", save_path=save_plots, action=action, mouse_log=mouse_description, ax1lim=(20,20), ax2lim=(20,50))
    day_comp_scatter(all_rates, day, Phases, 1, unit="rate", save_path=False, action=action, mouse_log=mouse_description)

plt.show()

## 4. Daily Summary Bar Plots
Uncomment the plots you want to generate.

In [ ]:

# visualization or saving the file
sav_e = False

days_to_plot = Days

plot_daily_summary_bar(all_amplitudes, days_to_plot, unit="amplitude", action=action, save_path=sav_e)
plot_daily_summary_bar(all_durations, days_to_plot, unit="duration", action=action, save_path=sav_e)
plot_daily_summary_bar(all_widths, days_to_plot, unit="width", action=action, save_path=sav_e)

# plot_daily_summary_bar(all_aucs, days_to_plot, unit="AUC", action=action, save_path=False)
# plot_daily_summary_bar(all_rates, days_to_plot, unit="Event Rate (per min)", action=action, save_path=False)

plt.show()

## 5. Daily Traces

In [ ]:
days_to_plot = Days

plot_daily_traces_separated(all_traces, mouse_description, days_to_plot, time_axis, zscore, action, save_path=False, phases=Phases)
plot_daily_traces(all_traces, mouse_description, days_to_plot, time_axis, zscore, action, save_path=False)
plot_mouse_daily_traces(all_traces, days_to_plot, time_axis, zscore, action, save_path=False)

plt.show()

## 6. Heatmaps
Generate a positional heatmap for a specific mouse/session 

In [ ]:
from dataloader import load_session_data, positional
from session_dict import barnes_session_dict
import matplotlib.pyplot as plt

# 1. Find the specific session you want (e.g., Mouse MHF200 on Day 1, Phase 1)
my_session = next(s for s in barnes_session_dict if s['id'] == 'MHF200' and s['day'] == 'D1' and s['phase'] == 'PH1')

# 2. Load just that one session's data 
data = load_session_data(my_session, save_file=False)

# 3. Generate the heatmap!
heatmap, escape_coords = positional(data['location'], data['signal_trace'], my_session)
plt.show()